# Semana 6 — Actividad antes de clase
## Evaluacion de modelos supervisados: el pipeline completo

**Curso:** Fundamentos de IA y Machine Learning — ULACIT  
**Instrucciones:** Ejecuten cada celda en orden (boton ▶ o `Shift+Enter`) y lean los comentarios con atencion. No necesitan modificar nada, solo **ejecutar, observar y reflexionar**.

---

### Que van a ver en este notebook

Vamos a recorrer el pipeline completo de Machine Learning aplicado a un problema real de **diagnostico medico**: clasificar tumores de mama como malignos o benignos.

1. Cargar y explorar los datos
2. Preparar los datos (dividir en train/test)
3. Entrenar **3 modelos diferentes**
4. Evaluar cada modelo con **multiples metricas**
5. Comparar y decidir cual modelo es "mejor" (y por que la respuesta no es tan simple)

> **Reflexionen mientras ejecutan:** ¿Cual modelo elegirian si tuvieran que diagnosticar a un paciente real? ¿Por que?


---
## Paso 1: Importar librerias

Estas son las herramientas que usaremos. Todas vienen preinstaladas en Google Colab.


In [ ]:
# Librerias para datos y visualizacion
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Librerias de Machine Learning
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# Librerias de evaluacion
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay,
    RocCurveDisplay
)

print("Todas las librerias cargadas correctamente.")


---
## Paso 2: Cargar y explorar los datos

Usaremos el dataset **Breast Cancer Wisconsin**, incluido en scikit-learn. Contiene 569 muestras de tumores con 30 caracteristicas medidas por imagenes digitalizadas (radio, textura, perimetro, area, suavidad, etc.).

**Variable objetivo:** Diagnostico — Maligno (0) o Benigno (1)


In [ ]:
# Cargar el dataset
datos = load_breast_cancer()
df = pd.DataFrame(datos.data, columns=datos.feature_names)
df['diagnostico'] = datos.target

# Explorar las dimensiones
print(f"Tamano del dataset: {df.shape[0]} muestras, {df.shape[1] - 1} variables")
print(f"\nDistribucion de la variable objetivo:")
print(f"  Benigno (1): {(df['diagnostico'] == 1).sum()} muestras ({(df['diagnostico'] == 1).mean():.1%})")
print(f"  Maligno (0): {(df['diagnostico'] == 0).sum()} muestras ({(df['diagnostico'] == 0).mean():.1%})")
print(f"\nPrimeras 5 filas del dataset:")
df.head()


### Visualizacion de la distribucion de clases

Observen que las clases **no estan perfectamente balanceadas**: hay mas tumores benignos que malignos. Esto es comun en problemas reales y afecta directamente como debemos evaluar el modelo.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribucion de clases
colores = ['#2d1b4e', '#00d4ff']
conteos = df['diagnostico'].value_counts().sort_index()
axes[0].bar(['Maligno (0)', 'Benigno (1)'], conteos.values, color=colores)
axes[0].set_title('Distribucion de diagnosticos', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Cantidad de muestras')
for i, v in enumerate(conteos.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold', fontsize=12)

# Comparacion de una variable clave por diagnostico
maligno = df[df['diagnostico'] == 0]['mean radius']
benigno = df[df['diagnostico'] == 1]['mean radius']
axes[1].hist(benigno, bins=25, alpha=0.7, label='Benigno', color='#00d4ff')
axes[1].hist(maligno, bins=25, alpha=0.7, label='Maligno', color='#2d1b4e')
axes[1].set_title('Distribucion de "radio promedio" por diagnostico', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Radio promedio del tumor')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.tight_layout()
plt.show()
print("Observen como los tumores malignos tienden a tener un radio mayor que los benignos.")
print("Esta es una de las 30 variables que los modelos usaran para hacer predicciones.")


---
## Paso 3: Preparar los datos

Ahora dividiremos los datos en **entrenamiento (80%) y prueba (20%)**. Luego escalaremos las variables.

**Punto critico:** Observen que el escalado (`StandardScaler`) se ajusta **solo con los datos de entrenamiento** y luego se aplica a los de prueba. Esto evita la **fuga de datos** (data leakage).


In [ ]:
# Separar variables (X) y objetivo (y)
X = df.drop('diagnostico', axis=1)
y = df['diagnostico']

# Dividir en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Datos de entrenamiento: {X_train.shape[0]} muestras")
print(f"Datos de prueba: {X_test.shape[0]} muestras")
print(f"\nDistribucion en entrenamiento — Benigno: {(y_train == 1).sum()}, Maligno: {(y_train == 0).sum()}")
print(f"Distribucion en prueba       — Benigno: {(y_test == 1).sum()}, Maligno: {(y_test == 0).sum()}")

# Escalar los datos (SOLO ajustar con train, aplicar a ambos)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform en train
X_test_scaled = scaler.transform(X_test)          # solo transform en test

print(f"\nDatos escalados correctamente.")
print(f"(El scaler se ajusto SOLO con datos de entrenamiento para evitar fuga de datos)")


---
## Paso 4: Entrenar tres modelos diferentes

Entrenaremos tres algoritmos distintos con los mismos datos:

| Modelo | Algoritmo | Caracteristica principal |
|---|---|---|
| **Modelo A** | Regresion logistica | Simple, interpretable, rapido |
| **Modelo B** | Random Forest | Ensamble de arboles, robusto |
| **Modelo C** | Red neuronal (MLP) | Complejo, captura patrones no lineales |

Cada modelo tiene fortalezas diferentes. La pregunta clave es: **¿cual es "mejor"?**


In [ ]:
# --- MODELO A: Regresion Logistica ---
modelo_a = LogisticRegression(random_state=42, max_iter=1000)
modelo_a.fit(X_train_scaled, y_train)
pred_a = modelo_a.predict(X_test_scaled)
prob_a = modelo_a.predict_proba(X_test_scaled)[:, 1]

# --- MODELO B: Random Forest ---
modelo_b = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_b.fit(X_train_scaled, y_train)
pred_b = modelo_b.predict(X_test_scaled)
prob_b = modelo_b.predict_proba(X_test_scaled)[:, 1]

# --- MODELO C: Red Neuronal (MLP) ---
modelo_c = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
modelo_c.fit(X_train_scaled, y_train)
pred_c = modelo_c.predict(X_test_scaled)
prob_c = modelo_c.predict_proba(X_test_scaled)[:, 1]

print("Los 3 modelos fueron entrenados exitosamente.")
print(f"\n  Modelo A (Regresion Logistica): entrenado")
print(f"  Modelo B (Random Forest):       entrenado")
print(f"  Modelo C (Red Neuronal MLP):    entrenado") 


---
## Paso 5: Evaluar — Matrices de confusion

La **matriz de confusion** es el punto de partida de toda evaluacion. Nos dice exactamente que tipo de errores comete cada modelo.

En este problema medico:
- **Falso Negativo (FN):** Tumor maligno clasificado como benigno → **el paciente se va a casa con cancer no detectado**
- **Falso Positivo (FP):** Tumor benigno clasificado como maligno → el paciente pasa por examenes adicionales innecesarios

> **Pregunta para reflexionar:** ¿Cual error es mas grave en este contexto?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

modelos = [
    ("Modelo A\nRegresion Logistica", pred_a),
    ("Modelo B\nRandom Forest", pred_b),
    ("Modelo C\nRed Neuronal", pred_c)
]

for ax, (nombre, pred) in zip(axes, modelos):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Maligno', 'Benigno'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(nombre, fontsize=13, fontweight='bold')

plt.suptitle('Matrices de confusion de los 3 modelos', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Interpretar en texto
for nombre, pred in [("Modelo A", pred_a), ("Modelo B", pred_b), ("Modelo C", pred_c)]:
    cm = confusion_matrix(y_test, pred)
    print(f"{nombre}:")
    print(f"  Falsos Negativos (cancer no detectado): {cm[0][1]}")
    print(f"  Falsos Positivos (falsa alarma):        {cm[1][0]}")
    print()


---
## Paso 6: Comparar metricas de los 3 modelos

Ahora calculamos **todas las metricas principales** para cada modelo y las ponemos lado a lado.

Recuerden:
- **Accuracy:** Proporcion total de aciertos (puede ser enganosa con datos desbalanceados)
- **Precision:** De los que el modelo dijo "maligno", cuantos realmente lo eran
- **Recall:** De todos los tumores malignos reales, cuantos detecto el modelo
- **F1-score:** Balance entre precision y recall
- **AUC-ROC:** Capacidad discriminativa global del modelo


In [ ]:
# Calcular metricas para los 3 modelos
resultados = []

for nombre, pred, prob in [
    ("Modelo A (Reg. Logistica)", pred_a, prob_a),
    ("Modelo B (Random Forest)", pred_b, prob_b),
    ("Modelo C (Red Neuronal)", pred_c, prob_c)
]:
    resultados.append({
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, pos_label=0),
        'Recall': recall_score(y_test, pred, pos_label=0),
        'F1-score': f1_score(y_test, pred, pos_label=0),
        'AUC-ROC': roc_auc_score(y_test, prob)
    })

df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.set_index('Modelo')

# Formatear como porcentajes
print("=" * 75)
print("COMPARACION DE METRICAS — DETECCION DE TUMORES MALIGNOS")
print("=" * 75)
print()
print(df_resultados.round(4).to_string())
print()
print("-" * 75)

# Identificar "ganadores" por metrica
for metrica in ['Accuracy', 'Precision', 'Recall', 'F1-score', 'AUC-ROC']:
    mejor = df_resultados[metrica].idxmax()
    valor = df_resultados[metrica].max()
    print(f"  Mejor en {metrica:12s}: {mejor} ({valor:.4f})")


### Visualizacion comparativa

Veamos las metricas de forma visual para facilitar la comparacion.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grafico de barras comparativo
metricas = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'AUC-ROC']
x = np.arange(len(metricas))
width = 0.25
colores_barras = ['#2d1b4e', '#00d4ff', '#7c3aed']

for i, (modelo, color) in enumerate(zip(df_resultados.index, colores_barras)):
    valores = df_resultados.loc[modelo, metricas].values
    axes[0].bar(x + i * width, valores, width, label=modelo.split('(')[0].strip(), color=color, alpha=0.85)

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(metricas, rotation=15)
axes[0].set_ylim(0.8, 1.02)
axes[0].set_title('Comparacion de metricas por modelo', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_ylabel('Valor de la metrica')
axes[0].grid(axis='y', alpha=0.3)

# Curvas ROC
for nombre, prob, color in [
    ("Modelo A", prob_a, '#2d1b4e'),
    ("Modelo B", prob_b, '#00d4ff'),
    ("Modelo C", prob_c, '#7c3aed')
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    axes[1].plot(fpr, tpr, color=color, linewidth=2, label=f'{nombre} (AUC={auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Azar (AUC=0.5)')
axes[1].set_title('Curvas ROC', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


---
## Paso 7: Detectar overfitting

Ahora comparemos el rendimiento de cada modelo en **entrenamiento vs. prueba**. Si un modelo tiene un rendimiento mucho mejor en entrenamiento que en prueba, esta **memorizando** en lugar de aprender — esto se llama **overfitting**.


In [ ]:
# Calcular accuracy en train y test para cada modelo
print("=" * 65)
print("DETECCION DE OVERFITTING: TRAIN vs. TEST")
print("=" * 65)
print()
print(f"{'Modelo':<30s} {'Train':>10s} {'Test':>10s} {'Brecha':>10s}")
print("-" * 65)

for nombre, modelo, X_tr, X_te in [
    ("Modelo A (Reg. Logistica)", modelo_a, X_train_scaled, X_test_scaled),
    ("Modelo B (Random Forest)", modelo_b, X_train_scaled, X_test_scaled),
    ("Modelo C (Red Neuronal)", modelo_c, X_train_scaled, X_test_scaled)
]:
    acc_train = accuracy_score(y_train, modelo.predict(X_tr))
    acc_test = accuracy_score(y_test, modelo.predict(X_te))
    brecha = acc_train - acc_test
    alerta = " ⚠ Posible overfitting" if brecha > 0.05 else ""
    print(f"{nombre:<30s} {acc_train:>9.2%} {acc_test:>9.2%} {brecha:>9.2%}{alerta}")

print()
print("Si la brecha entre train y test es grande (>5%), puede haber overfitting.")
print("El modelo estaria memorizando los datos de entrenamiento en vez de aprender")
print("patrones generalizables a datos nuevos.")


---
## Paso 8: Validacion cruzada

Para tener una evaluacion mas robusta, usamos **validacion cruzada de 5 folds**. Esto divide los datos en 5 partes, entrena 5 veces usando diferentes combinaciones, y reporta la media y desviacion estandar.

Una **desviacion estandar alta** indica que el modelo es inestable (su rendimiento varia mucho dependiendo de que datos le toquen).


In [ ]:
print("=" * 65)
print("VALIDACION CRUZADA (5-Fold) — ACCURACY")
print("=" * 65)
print()

for nombre, modelo in [
    ("Modelo A (Reg. Logistica)", LogisticRegression(random_state=42, max_iter=1000)),
    ("Modelo B (Random Forest)", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("Modelo C (Red Neuronal)", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42))
]:
    scores = cross_val_score(modelo, X_train_scaled, y_train, cv=5, scoring='accuracy')
    print(f"{nombre}:")
    print(f"  Scores por fold: {[f'{s:.4f}' for s in scores]}")
    print(f"  Media: {scores.mean():.4f}  |  Desviacion estandar: {scores.std():.4f}")
    print()

print("Una desviacion estandar baja indica que el modelo es estable y consistente.")


---
## Paso 9: La decision — ¿Cual modelo elegimos?

Ahora viene la parte mas importante. Miremos todo junto y pensemos como un **equipo de ciencia de datos en un hospital**.


In [ ]:
print("=" * 75)
print("RESUMEN EJECUTIVO: ¿CUAL MODELO ELEGIMOS?")
print("=" * 75)
print()
print(df_resultados.round(4).to_string())
print()
print("=" * 75)
print()
print("EN ESTE PROBLEMA MEDICO:")
print()
print("  Un FALSO NEGATIVO significa decirle a un paciente con cancer que")
print("  esta sano. Podria costarle la vida.")
print()
print("  Un FALSO POSITIVO significa decirle a un paciente sano que podria")
print("  tener cancer. Genera ansiedad pero se resuelve con mas examenes.")
print()
print("  Por lo tanto, en este contexto, RECALL es la metrica prioritaria")
print("  porque necesitamos detectar la mayor cantidad de tumores malignos")
print("  posible, aun si eso genera algunas falsas alarmas adicionales.")
print()
print("-" * 75)

# Identificar el mejor modelo por recall
mejor_recall = df_resultados['Recall'].idxmax()
valor_recall = df_resultados['Recall'].max()
print(f"\n  >>> RECOMENDACION: {mejor_recall}")
print(f"      Recall: {valor_recall:.4f}")
print(f"      AUC-ROC: {df_resultados.loc[mejor_recall, 'AUC-ROC']:.4f}")
print(f"      F1-score: {df_resultados.loc[mejor_recall, 'F1-score']:.4f}")
print()
print("  Este modelo detecta la mayor proporcion de tumores malignos,")
print("  que es lo mas critico en un diagnostico medico.")
print()
print("=" * 75)
print()
print("  PERO... ¿y si el problema no fuera medico?")
print("  ¿Y si estuvieramos filtrando spam? ¿O aprobando prestamos?")
print("  La metrica prioritaria cambiaria. No existe una respuesta universal.")
print("  La seleccion de la metrica correcta es una DECISION DE NEGOCIO.")
print()
print("  Esto es exactamente lo que veremos en la clase de esta semana.")


---
## Preguntas de reflexion

Despues de ejecutar todo el notebook, reflexionen sobre lo siguiente antes de la clase:

1. **¿Por que el modelo con mayor accuracy no necesariamente es el mejor?** ¿En que escenarios podria ser enganoso confiar solo en accuracy?

2. **¿Como cambiaria su decision si en lugar de diagnostico medico estuvieran aprobando prestamos bancarios?** ¿Que tipo de error seria mas costoso?

3. **¿Notaron diferencias entre el rendimiento en entrenamiento y en prueba?** ¿Que podria explicar esas diferencias?

4. **¿Por que es importante hacer validacion cruzada** en lugar de evaluar con una sola division train/test?

5. **¿Que pasaria si usaran este modelo con datos de un hospital diferente?** ¿Funcionaria igual de bien?

---

> Estas reflexiones seran el punto de partida de nuestra discusion en clase. Nos vemos en la sesion de la Semana 6.
